# MOSS-Image LoRA 学習ノートブック (ComfyUI互換・完全自動・簡単テンプレート) v1.0

**このノートブックの構造**: あなたが操作するのは**最初の3セルだけ** (トークン入力・Drive許可・キュー確認)。設定ファイル (jobs.json) は**不要**です。
`MyDrive/moss_image/queue/` にZIPファイルを置くだけで自動検出・連続学習し、完了したZIPは `done/` へ自動移動、完成したLoRAは `outputs/` に保存されます。

**モデル構成** (MOSS-Image, OpenMOSS):
- DiT: 5B single-stream DiT (42 blocks) / Rectified Flow
- Text Encoder: Qwen3.5-9B (bf16)
- VAE: FLUX.2 VAE (32ch, 8x spatial downscale)
- 学習スクリプト: `kohya-ss/sd-scripts` の公式対応 `moss_image_train_network.py`
- 出力: `networks.lora_generic` の `.safetensors`。**ComfyUI で変換なしにそのまま読込可能**

**フォルダ構成** (Google Drive上に自動作成されます):
- `MyDrive/moss_image/queue/` : 学習待ちのデータセットZIPを配置
- `MyDrive/moss_image/done/` : 学習が完了したZIPが自動移動
- `MyDrive/moss_image/outputs/` : 完成LoRA + メタ情報JSONが出力

**事前準備** (初回のみ):
1. **HuggingFaceトークン** — `Comfy-Org/MOSS-Image` と `OpenMOSS-Team/MOSS-Image-Base` は公開リポジトリのため必須ではありませんが、レート制限対策として設定を推奨します。
2. **データセットZIP** — 画像15〜30枚 + 同名 `.txt` キャプション（自然文推奨）を `MyDrive/moss_image/queue/` に配置。ZIP内に `trigger.txt` を入れるとトリガーワードを指定できます（なければZIP名から自動生成）。

**学習パラメータ** (MOSS-Image 公式ガイドの初期値):
- 解像度: 1024px（bucket 64px 刻み、512〜2048）
- 学習率 (lr): 1e-4 / ランク (rank): 16 / alpha: 1.0（sd-scripts デフォルト）
- ステップ数: 画像枚数から自動決定（`calc_steps`。実測調整前提の初期値）
- タイムステップ: `sigma` + logit-normal(-1.09, 0.95) + shift 1.0（事前学習分布の再現）
- 事前キャッシュ: `cache_latents.py --moss_image` / `cache_text_encoder_outputs.py --moss_image`

**必要なランタイム**:
- **L4 (24GB) または A100 (40GB)** を推奨。
- **T4 は不可**: MOSS-Image の DiT / Text Encoder は bf16 専用で、T4 (sm75) は bf16 非対応です。このノートブックはT4を検出すると明示的に停止します（無理に実行してOOM/カーネルエラーにしないため）。

**実行手順**: 最初の3セルを実行 → メニュー「ランタイム」→「以降のセルを実行」。あとは放置でOKです。学習時間は画像枚数・GPU・ステップ数に依存します。

## 1. あなたの操作 (ここだけ・最初に全部済ませる)

### 1-1. HuggingFaceトークン

Colabの左サイドバーにある **🔑（シークレット）** アイコンから `HF_TOKEN` を登録しておくか、未登録の場合は実行時に入力プロンプトが表示されます。
MOSS-Image のモデルは公開されていますが、設定しておくとレート制限や将来のgated化に備えられます。

In [ ]:
import getpass, os

token = None
# 1. Colabのシークレット機能 (🔑) から取得を試みる
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

# 2. 環境変数に既にあるか確認
if not token:
    token = os.environ.get("HF_TOKEN")

# 3. なければ入力プロンプトを表示 (非表示入力)
if not token:
    token = getpass.getpass("HuggingFace Access Token を入力してください (hf_..., 空Enterでスキップ): ").strip()

if token:
    os.environ["HF_TOKEN"] = token
    print("トークン: OK (設定済み)")
else:
    print("トークン: 未設定 (公開モデルのため続行可能)")

### 1-2. Google Driveマウント

ポップアップでGoogleアカウント選択・許可が出たら「許可」をクリック。画面の陰に隠れやすいので注意。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("DriveマウントOK")

### 1-3. キュー確認 (設定ファイル不要・queueに入れるだけ)

`MyDrive/moss_image/queue/` に置かれたZIPファイルを順番に学習します。
正常終了したZIPは自動的に `MyDrive/moss_image/done/` に移動します。

- トリガーワードは ZIP 内の `trigger.txt`（1行目）→ なければ ZIP ファイル名から自動生成
- `MyDrive/moss_image/` 直下に置いたZIPも、実行時に自動で `queue/` へ移動します

In [ ]:
# ============================================================
# 1-3. キュー確認 (設定ファイル不要・queueに入れるだけ)
# フォルダ構成:
#  - 学習待ち: MyDrive/moss_image/queue/
#  - 学習完了: MyDrive/moss_image/done/
#  - LoRA出力: MyDrive/moss_image/outputs/
# ============================================================
import glob, os, re, shutil, zipfile

BASE_DIR   = "/content/drive/MyDrive/moss_image"
QUEUE_DIR  = os.path.join(BASE_DIR, "queue")
DONE_DIR   = os.path.join(BASE_DIR, "done")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

for d in (QUEUE_DIR, DONE_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

# BASE_DIR 直下に置かれた ZIP も queue へ自動移動
moved = 0
for z in glob.glob(os.path.join(BASE_DIR, "*.zip")):
    shutil.move(z, os.path.join(QUEUE_DIR, os.path.basename(z)))
    moved += 1
if moved:
    print(f"📥 {moved} 個のZIPを queue/ へ移動しました")

# 学習パラメータ (MOSS-Image 公式ガイドの初期値を基準)
RESOLUTION = 1024
LR         = 1e-4
RANK       = 16
STEPS      = None  # 未指定なら calc_steps() が画像枚数から自動決定

def sanitize_trigger(value):
    # パス区切り・制御文字・空白だけを置換する (日本語トリガーはそのまま許可)
    value = re.sub(r'[/\\:*?"<>|\x00-\x1f]', "_", str(value)).strip()
    value = re.sub(r"\s+", "_", value)
    value = value.strip("._-")
    return value or "moss"

def extract_trigger_from_zip(zip_path):
    # ZIP内の trigger.txt を優先。なければZIP名から生成する。
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            for name in z.namelist():
                if os.path.basename(name).lower() == "trigger.txt":
                    text = z.read(name).decode("utf-8", errors="ignore").strip()
                    first = next((ln.strip() for ln in text.splitlines() if ln.strip()), "")
                    if first:
                        return sanitize_trigger(first)
    except Exception:
        pass
    base = os.path.basename(zip_path)
    if base.lower().endswith(".zip"):
        base = base[:-4]
    clean = re.sub(r"[_\-\s]*\d{8}(?:[_\-\s]*\d+)?$", "", base)
    clean = re.sub(r"[_\-\s]*\d+$", "", clean).strip(" _-")
    clean = sanitize_trigger(clean)
    return clean if clean.endswith("_style") else f"{clean}_style"

zip_files = sorted(glob.glob(os.path.join(QUEUE_DIR, "*.zip")))
assert zip_files, (
    "学習待ちのZIPが見つかりません。\n"
    f'Google Drive の "{QUEUE_DIR}" に、画像+キャプションのZIPを配置してください。'
)

JOBS = []
for z in zip_files:
    JOBS.append({
        "trigger": extract_trigger_from_zip(z),
        "zip": z,
        "steps": STEPS,
        "lr": LR,
        "rank": RANK,
    })

# trigger の重複を検出 (outputs の上書きを防ぐ)
seen = {}
for j in JOBS:
    seen.setdefault(j["trigger"], []).append(os.path.basename(j["zip"]))
dups = {k: v for k, v in seen.items() if len(v) > 1}
if dups:
    print("⚠️ trigger名が重複しています。ZIP内の trigger.txt で別名を指定してください:")
    for k, v in dups.items():
        print(f"   {k}: {v}")

print("=" * 64)
print(f"🚀 キュー内の学習対象: 計 {len(JOBS)} 件")
print(f"📂 キューフォルダ: {QUEUE_DIR}")
print(f"📦 完了移動先: {DONE_DIR}")
print(f"💾 出力先: {OUTPUT_DIR}")
print(f"⚙️ 設定: 解像度 {RESOLUTION}px / ステップ: {STEPS if STEPS else 'AUTO'} / lr: {LR} / rank: {RANK}")
print("-" * 64)
for i, j in enumerate(JOBS, 1):
    print(f" [{i}] trigger: '{j['trigger']}' | zip: '{os.path.basename(j['zip'])}'")
print("=" * 64)
print("問題なければ以降のセルを実行してください。\n")

## 2. 環境構築 (自動・一度実行すればスキップ)

`kohya-ss/sd-scripts` を clone し、`requirements.txt` をインストールします（数分かかります）。
あわせて GPU の bf16 対応と VRAM を検査し、学習プロファイルを自動選択します。

- `VRAM >= 36GB` → **bf16** (DiT / Text Encoder とも bf16、スワップなし)
- `18GB <= VRAM < 36GB` → **ConvRot INT8** (DiT / Text Encoder とも事前量子化ファイル)
- `VRAM < 18GB` (bf16対応GPU) → **ConvRot INT8 + blocks_to_swap 36**
- **bf16非対応GPU (T4等)** → 実行不可として明示停止

In [ ]:
import importlib.util, os, subprocess, sys
import torch

print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__)

assert torch.cuda.is_available(), (
    "GPUランタイムが見つかりません。メニュー「ランタイム」→「ランタイムのタイプを変更」で "
    "L4 (24GB) または A100 (40GB) を選択してください。"
)

SD_DIR = "/content/sd-scripts"
SD_REPO = "https://github.com/kohya-ss/sd-scripts.git"
SETUP_MARK = "/content/.moss_env_done"

if not os.path.exists(SETUP_MARK):
    if not os.path.isdir(os.path.join(SD_DIR, ".git")):
        print("sd-scripts を clone します...")
        subprocess.run(["git", "clone", "--depth", "1", SD_REPO, SD_DIR], check=True)
    else:
        print("sd-scripts: clone 済み")

    print("依存パッケージをインストールします (数分かかります)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(SD_DIR, "requirements.txt")],
        cwd=SD_DIR,
        check=True,
    )
    open(SETUP_MARK, "w").write("done")
    print("環境構築完了")
else:
    print("環境構築: 済み (スキップ)")

# accelerate のデフォルト設定 (未設定でも launch は動くが、警告・対話プロンプトを避ける)
subprocess.run(
    ["accelerate", "config", "default"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# triton は ConvRot INT8 の融合カーネルで使う (なくても bf16 フォールバックで動作)
TRITON_OK = importlib.util.find_spec("triton") is not None
if not TRITON_OK:
    print("triton が見つかりません。インストールを試みます...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=False)
    TRITON_OK = r.returncode == 0 and importlib.util.find_spec("triton") is not None

props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / 1e9
GPU_NAME = torch.cuda.get_device_name(0)
BF16_OK = torch.cuda.is_bf16_supported()

print(
    f"GPU: {GPU_NAME} / VRAM: {VRAM_GB:.1f}GB / "
    f"bf16: {'OK' if BF16_OK else 'NG'} / triton: {'OK' if TRITON_OK else 'NG'}"
)

if not BF16_OK:
    PROFILE = "unsupported"
    print("⚠️ このGPUは bf16 非対応です (T4 など sm75 以下)。")
    print("   MOSS-Image の DiT と Text Encoder は bf16 専用のため、公式パイプラインは実行できません。")
    print("   ランタイムを L4 (24GB) または A100 (40GB) に変更してください。")
elif VRAM_GB >= 36:
    PROFILE = "bf16"
    print("→ 構成: bf16 DiT + bf16 Text Encoder / 事前キャッシュ / ブロックスワップなし")
elif VRAM_GB >= 18:
    PROFILE = "int8"
    print("→ 構成: ConvRot INT8 DiT + INT8 Text Encoder / 事前キャッシュ")
else:
    PROFILE = "int8_swap"
    print("→ 構成: ConvRot INT8 DiT + INT8 Text Encoder / blocks_to_swap 36")

## 3. モデルダウンロード (自動・DL済みならスキップ)

`Comfy-Org/MOSS-Image` から、プロファイルに応じた単一ファイルを取得します。

| プロファイル | DiT | Text Encoder | VAE | 合計 |
|---|---|---|---|---|
| bf16 | `moss_image_base_bf16` 10.2GB | `qwen3.5_9b_bf16` 19.3GB | `flux2-vae` 0.3GB | 約30GB |
| INT8 | `moss_image_base_int8_convrot` 5.4GB | `qwen3.5_9b_int8_convrot` 9.7GB | `flux2-vae` 0.3GB | 約15.5GB |

- 保存先は `/content/models/moss_image`（セッション終了で消えます）
- ダウンロード後の HF メタキャッシュは削除してディスクを節約します

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

MODEL_ROOT = "/content/models/moss_image"
REPO_ID = "Comfy-Org/MOSS-Image"

# 期待サイズ (bytes)。HF API の実サイズ。99%以上あればDL済みとみなす。
FILES_BF16 = [
    ("diffusion_models/moss_image_base_bf16.safetensors", 10164655808),
    ("text_encoders/qwen3.5_9b_bf16.safetensors", 19306312328),
]
FILES_INT8 = [
    ("diffusion_models/moss_image_base_int8_convrot.safetensors", 5413698958),
    ("text_encoders/qwen3.5_9b_int8_convrot.safetensors", 9748121532),
]
VAE_FILE = ("vae/flux2-vae.safetensors", 336211292)

if PROFILE == "unsupported":
    DIT_PATH = TE_PATH = VAE_PATH = None
    print("モデルDL: スキップ (このGPUでは実行不可)")
else:
    files = (FILES_BF16 if PROFILE == "bf16" else FILES_INT8) + [VAE_FILE]
    need_gb = sum(size for _, size in files) / 1e9 * 1.1 + 4.0
    free_gb = shutil.disk_usage("/content").free / 1e9
    print(f"ディスク空き: {free_gb:.1f}GB / 必要見込み: {need_gb:.1f}GB")
    assert free_gb >= need_gb, (
        f"ディスク容量が不足しています (空き {free_gb:.1f}GB < 必要 {need_gb:.1f}GB)。"
        " ランタイムのディスクを確認してください。"
    )

    os.makedirs(MODEL_ROOT, exist_ok=True)
    paths = {}
    for rel, size in files:
        dst = os.path.join(MODEL_ROOT, rel)
        if os.path.exists(dst) and os.path.getsize(dst) >= size * 0.99:
            print(f"既存: {rel} ({os.path.getsize(dst) / 1e9:.1f}GB)")
        else:
            print(f"DL中: {rel} ...")
            hf_hub_download(repo_id=REPO_ID, filename=rel, local_dir=MODEL_ROOT)
            got = os.path.getsize(dst) if os.path.exists(dst) else 0
            assert got >= size * 0.99, f"ダウンロードが不完全です: {rel} ({got} / {size} bytes)"
            print(f"  -> {got / 1e9:.1f}GB")
        paths[rel] = dst

    DIT_PATH = paths[files[0][0]]
    TE_PATH = paths[files[1][0]]
    VAE_PATH = paths[VAE_FILE[0]]

    # local_dir 内の HF メタキャッシュを削除 (実体ファイルは残る)
    cache_dir = os.path.join(MODEL_ROOT, ".cache")
    if os.path.isdir(cache_dir):
        shutil.rmtree(cache_dir, ignore_errors=True)

    print(f"DiT: {DIT_PATH}")
    print(f"TE : {TE_PATH}")
    print(f"VAE: {VAE_PATH}")
print("MODELS_DONE")

## 4. データ準備 (ZIP展開・キャプション検査・dataset config)

ZIP を `/content/dataset` に展開し、ラッパーフォルダを平坦化してからキャプションを検査します。

- 画像と同名の `.txt` をキャプションとして使用（自然文・改行可）
- キャプション欠落 → トリガーのみのキャプションを自動生成
- トリガーが未含なら先頭に `{trigger}, ` を付与（本文は壊さない）
- `moss_dataset.toml` を生成（bucket 64px、caption_dropout 0.05、multiline 有効）

※ krea2 のようなアニメタグの削除・並べ替えは行いません。MOSS-Image はLLM生成の自然文キャプションで学習されているためです。

In [ ]:
import glob, os, re, shutil, zipfile

DATASET_DIR = "/content/dataset"
TOML_PATH = "/content/moss_dataset.toml"
IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".bmp")

def calc_steps(n_imgs):
    # 画像枚数から学習ステップ数の初期値を決める (MOSS-Image 用。実測調整前提)
    if n_imgs <= 10:
        return 500
    if n_imgs <= 30:
        return 800
    if n_imgs <= 50:
        return 1000
    if n_imgs <= 100:
        return 1200
    return 1500

def _safe_extract(zip_path, dest):
    # Zip Slip 対策: 展開先の外へ出るエントリを拒否
    dest_real = os.path.realpath(dest)
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            name = info.filename
            if name.startswith("__MACOSX/") or name.endswith("/"):
                continue
            target = os.path.realpath(os.path.join(dest, name))
            if target != dest_real and not target.startswith(dest_real + os.sep):
                raise RuntimeError(f"ZIP内に不正なパスがあります: {name}")
            z.extract(info, dest)

def _flatten_all(dest):
    # サブフォルダを平坦化する。同名衝突時はフォルダ名を前置して画像/キャプションの対応を保つ。
    for _ in range(10):
        subdirs = [d for d in glob.glob(os.path.join(dest, "*")) if os.path.isdir(d)]
        if not subdirs:
            return
        for sub in subdirs:
            files = []
            for root, _dirs, names in os.walk(sub):
                for n in names:
                    if not n.startswith("."):
                        files.append(os.path.join(root, n))
            if not files:
                shutil.rmtree(sub, ignore_errors=True)
                continue
            conflict = any(os.path.exists(os.path.join(dest, os.path.basename(f))) for f in files)
            prefix = re.sub(r"[^0-9A-Za-z_\-]+", "_", os.path.basename(sub)) + "_" if conflict else ""
            for f in files:
                name = prefix + os.path.basename(f)
                dst = os.path.join(dest, name)
                stem, ext = os.path.splitext(name)
                n = 1
                while os.path.exists(dst):
                    dst = os.path.join(dest, f"{stem}_{n}{ext}")
                    n += 1
                shutil.move(f, dst)
            shutil.rmtree(sub, ignore_errors=True)

def _caption_path(image_path):
    return os.path.splitext(image_path)[0] + ".txt"

def _read_caption(path):
    # UTF-8 BOM / UTF-8 / CP932 の順で試す (日本語キャプション対策)
    data = open(path, "rb").read()
    for enc in ("utf-8-sig", "utf-8", "cp932"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            continue
    return data.decode("utf-8", errors="ignore")

def _ensure_trigger(caption, trigger):
    cap = caption.strip()
    if not cap:
        return trigger
    first = cap.split(",", 1)[0].strip().lower()
    if first == trigger.lower():
        return cap
    if trigger.lower() in cap.lower():
        return cap
    return f"{trigger}, {cap}"

def prepare_dataset(zip_path, trigger):
    assert os.path.exists(zip_path), f"ZIPが見つかりません: {zip_path}"
    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    os.makedirs(DATASET_DIR, exist_ok=True)
    _safe_extract(zip_path, DATASET_DIR)
    _flatten_all(DATASET_DIR)

    # trigger.txt は学習データではないので削除 (サブフォルダ内も含む)
    for root, _dirs, names in os.walk(DATASET_DIR):
        for n in names:
            if n.lower() == "trigger.txt":
                os.remove(os.path.join(root, n))

    imgs = sorted(
        p for p in glob.glob(os.path.join(DATASET_DIR, "*"))
        if os.path.isfile(p) and os.path.splitext(p)[1].lower() in IMAGE_EXTS
    )
    assert imgs, f"画像が見つかりません: {zip_path} (対応拡張子: {', '.join(IMAGE_EXTS)})"

    missing, empty, changed = [], [], 0
    for img in imgs:
        txt = _caption_path(img)
        if not os.path.exists(txt):
            missing.append(os.path.basename(img))
            open(txt, "w", encoding="utf-8").write(trigger + "\n")
            continue
        raw = _read_caption(txt).strip()
        if not raw:
            empty.append(os.path.basename(img))
        fixed = _ensure_trigger(raw, trigger)
        if fixed != raw:
            changed += 1
        open(txt, "w", encoding="utf-8").write(fixed + "\n")

    caps = sorted(glob.glob(os.path.join(DATASET_DIR, "*.txt")))
    if len(imgs) < 10:
        print(f"⚠️ 画像が {len(imgs)} 枚です。MOSS-Image の画風LoRAは 15〜30 枚を推奨します。")
    print(f"データセット: 画像 {len(imgs)} 枚 / キャプション {len(caps)} 個 / trigger='{trigger}'")
    print(f"  キャプション補完: 欠落 {len(missing)} / 空 {len(empty)} / trigger付与 {changed}")
    if missing:
        print("  欠落の例:", ", ".join(missing[:5]))
    sample = _read_caption(_caption_path(imgs[0])).strip()
    print("  例:", sample[:160] + ("..." if len(sample) > 160 else ""))
    return len(imgs)

def make_dataset_toml(res):
    toml = f'''[general]
enable_bucket = true
bucket_no_upscale = false
min_bucket_reso = 512
max_bucket_reso = 2048
bucket_reso_steps = 64
caption_extension = ".txt"

[[datasets]]
resolution = [{res}, {res}]
batch_size = 1
enable_bucket = true

  [[datasets.subsets]]
  image_dir = "{DATASET_DIR}"
  num_repeats = 1
  caption_extension = ".txt"
  caption_dropout_rate = 0.05
  enable_multiline_caption = true
'''
    open(TOML_PATH, "w", encoding="utf-8").write(toml)
    print(f"dataset config 書込OK: {TOML_PATH} (resolution {res}px)")
    return TOML_PATH

print("関数定義OK (prepare_dataset / make_dataset_toml / calc_steps)")

## 5. 学習・納品 (事前キャッシュ → moss_image_train_network.py → Drive保存)

各ジョブで以下を実行します。

1. `tools/cache_latents.py --moss_image` で FLUX.2 VAE の潜在変数をディスクに事前キャッシュ
2. `tools/cache_text_encoder_outputs.py --moss_image` で Qwen3.5-9B の出力を事前キャッシュ
   （学習時に Text Encoder を GPU から解放でき、VRAM を大幅に節約）
3. `moss_image_train_network.py` で LoRA 学習
   - `networks.lora_generic` / dim 16 / DiT のみ
   - `--timestep_sampling sigma --weighting_scheme logit_normal --logit_mean -1.09 --logit_std 0.95 --discrete_flow_shift 1.0`
   - `--save_every_n_steps 100 --save_last_n_steps 2`
4. 最終 LoRA + 直近チェックポイントを `MyDrive/moss_image/outputs/` へコピーし、メタ情報JSONも保存

In [ ]:
import datetime, glob, json, os, re, shlex, shutil, subprocess, sys, time

def run_cmd(cmd, cwd, label):
    # シェルを介さずリストで実行 (ZIP名・トリガーのメタ文字事故を防ぐ)
    print(f"\n$ {' '.join(shlex.quote(str(c)) for c in cmd)}")
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=cwd)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"{label} が失敗しました (exit {proc.returncode}, {elapsed:.0f}s)")
    print(f"{label}: OK ({elapsed:.0f}s)")

def precache(toml_path):
    cache_latents_cmd = [
        sys.executable, "tools/cache_latents.py",
        "--moss_image",
        f"--vae={VAE_PATH}",
        f"--dataset_config={toml_path}",
        "--cache_latents_to_disk",
        "--vae_batch_size=1",
        "--mixed_precision=bf16",
    ]
    run_cmd(cache_latents_cmd, SD_DIR, "cache_latents")

    cache_te_cmd = [
        sys.executable, "tools/cache_text_encoder_outputs.py",
        "--moss_image",
        f"--text_encoder={TE_PATH}",
        f"--dataset_config={toml_path}",
        "--cache_text_encoder_outputs_to_disk",
        "--text_encoder_batch_size=1",
        "--mixed_precision=bf16",
    ]
    run_cmd(cache_te_cmd, SD_DIR, "cache_text_encoder_outputs")

def build_train_cmd(trigger, steps, out_dir, toml_path):
    cmd = [
        "accelerate", "launch", "--num_cpu_threads_per_process", "1",
        "moss_image_train_network.py",
        f"--pretrained_model_name_or_path={DIT_PATH}",
        f"--text_encoder={TE_PATH}",
        f"--vae={VAE_PATH}",
        f"--dataset_config={toml_path}",
        f"--output_dir={out_dir}",
        f"--output_name={trigger}_moss_lora",
        "--save_model_as=safetensors",
        "--network_module=networks.lora_generic",
        f"--network_dim={RANK}",
        "--network_train_unet_only",
        f"--learning_rate={LR}",
        "--optimizer_type=AdamW8bit",
        "--lr_scheduler=constant",
        "--timestep_sampling=sigma",
        "--weighting_scheme=logit_normal",
        "--logit_mean=-1.09",
        "--logit_std=0.95",
        "--discrete_flow_shift=1.0",
        f"--max_train_steps={steps}",
        "--save_every_n_steps=100",
        "--save_last_n_steps=2",
        "--mixed_precision=bf16",
        "--gradient_checkpointing",
        "--cache_latents",
        "--cache_latents_to_disk",
        "--cache_text_encoder_outputs",
        "--cache_text_encoder_outputs_to_disk",
        "--max_data_loader_n_workers=2",
        "--persistent_data_loader_workers",
        "--seed=42",
    ]
    if PROFILE == "int8_swap":
        cmd.append("--blocks_to_swap=36")
    return cmd

def train(trigger, steps, out_dir, toml_path):
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(out_dir, exist_ok=True)
    run_cmd(build_train_cmd(trigger, steps, out_dir, toml_path), SD_DIR, f"train:{trigger}")
    files = sorted(glob.glob(os.path.join(out_dir, "*.safetensors")))
    assert files, f"学習出力がありません: {out_dir}"
    print(f"学習出力: {len(files)} ファイル")
    for f in files:
        print(f"  {os.path.basename(f)} ({os.path.getsize(f) / 1e6:.0f}MB)")
    return files

def deliver(trigger, steps, out_dir, elapsed=None):
    files = sorted(glob.glob(os.path.join(out_dir, "*.safetensors")))
    assert files, f"納品対象がありません: {out_dir}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    jst = datetime.timezone(datetime.timedelta(hours=9))
    dstr = datetime.datetime.now(jst).strftime("%Y%m%d")
    delivered = []
    for src in files:
        base = os.path.basename(src)
        m = re.search(r"-step(\d+)\.safetensors$", base)
        label = int(m.group(1)) if m else steps
        dst_name = f"{trigger}_{dstr}_{label:04d}steps_lr{LR}_dim{RANK}.safetensors"
        dst = os.path.join(OUTPUT_DIR, dst_name)
        shutil.copy2(src, dst)
        delivered.append(dst_name)
        print(f"Drive保存: {dst_name} ({os.path.getsize(dst) / 1e6:.0f}MB)")

    meta = {
        "trigger": trigger,
        "base_model": "Comfy-Org/MOSS-Image",
        "profile": PROFILE,
        "resolution": RESOLUTION,
        "steps": steps,
        "lr": LR,
        "rank": RANK,
        "optimizer": "AdamW8bit",
        "timestep_sampling": "sigma",
        "created_at": datetime.datetime.now(jst).isoformat(),
        "files": delivered,
    }
    if elapsed is not None:
        meta["elapsed_sec"] = round(elapsed, 1)
    meta_path = os.path.join(OUTPUT_DIR, f"{trigger}_{dstr}_meta.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f"メタ情報: {os.path.basename(meta_path)}")
    return delivered

print("関数定義OK (precache / build_train_cmd / train / deliver)")

## 6. 全ジョブ実行 (自動・本丸)

JOBSの全データセットを順番に: **展開 → キャプション検査 → 事前キャッシュ → 学習 → Drive保存** します。
- 進捗は `JOB 1/4: trigger` → `JOB_DONE: trigger` の流れ
- 1ジョブがエラーしても**残りは続行**し、最後に `FAILED: [...]` を表示
- **再実行すると、完了済みジョブはスキップされ、失敗ジョブだけ最初からクリーンに学習し直します**
- 学習中の loss はリアルタイム表示
- **ブラウザのタブを閉じないこと** (切断すると中断されます)

In [ ]:
import os, shutil, time, traceback

if PROFILE == "unsupported":
    print("=" * 64)
    print("STOP: このGPUは bf16 非対応のため、MOSS-Image を学習できません。")
    print(f"GPU: {GPU_NAME} / VRAM: {VRAM_GB:.1f}GB")
    print("ランタイムを L4 (24GB) または A100 (40GB) に変更して、最初から実行してください。")
    print("=" * 64)
else:
    DONE = "/content/jobs_done"
    os.makedirs(DONE, exist_ok=True)

    N = len(JOBS)
    failed = []
    t_all = time.time()

    for i, job in enumerate(JOBS, 1):
        trig = job["trigger"]
        zip_ = job["zip"]
        marker = os.path.join(DONE, f"{trig}.done")

        if os.path.exists(marker):
            print(f"\nSKIP: {trig} は完了済み (再実行時はスキップ)")
            continue

        print(f"\n{'=' * 64}\nJOB {i}/{N}: {trig} ({os.path.basename(zip_)})\n{'=' * 64}")
        out_dir = f"/content/moss_output/{trig}"
        t0 = time.time()
        try:
            n_imgs = prepare_dataset(zip_, trig)
            steps = job.get("steps") or STEPS or calc_steps(n_imgs)
            print(f"→ ステップ数: {steps} (画像 {n_imgs} 枚)")

            toml_path = make_dataset_toml(RESOLUTION)
            precache(toml_path)
            train(trig, steps, out_dir, toml_path)
            deliver(trig, steps, out_dir, elapsed=time.time() - t0)

            # 正常終了したZIPを done フォルダへ移動
            if os.path.exists(zip_):
                done_dest = os.path.join(DONE_DIR, os.path.basename(zip_))
                shutil.move(zip_, done_dest)
                print(f"📦 完了ZIPを移動: {os.path.basename(zip_)} -> {DONE_DIR}")

            open(marker, "w").write("done")
            print(f"JOB_DONE: {trig} ({i}/{N}, {time.time() - t0:.0f}s)")
        except Exception as e:
            # エラー時は記録のみ。再実行すると未完了ジョブだけ最初からやり直す
            print(f"!! JOB_ERROR: {trig} -> {type(e).__name__}: {e}")
            traceback.print_exc()
            failed.append(trig)

    print("\n" + "=" * 64)
    if failed:
        print(f"FAILED: {len(failed)}ジョブ -> {failed}")
        print("→ このセルを再実行すると、失敗ジョブだけクリーンにやり直します")
    else:
        print(f"ALL_JOBS_DONE 🎉 全ジョブ完了! ({(time.time() - t_all) / 60:.1f}分)")
    print("=" * 64)

## 完了! 次のステップ

1. 各LoRAは `MyDrive/moss_image/outputs/` に保存されています（最終 + 直近チェックポイント + メタ情報JSON）
2. サーバーへ配置: `~/ComfyUI/models/loras/` にscp、またはファイルを渡してもらえればこちらで配置します
3. ComfyUI での推奨: MOSS-Image 用の LoRA ローダーにそのまま読み込み / strength 1.0 / プロンプトは `{トリガー}, ...` 形式
   - 変換は不要です（`networks.lora_generic` の q/k/v 別キーを ComfyUI が fused qkv にマップ）

**トラブルシュート**:
- `T4` / bf16 NG → L4 または A100 に変更（MOSS-Image は bf16 専用）
- OOM (L4) → ノートブックは自動で INT8 を選択します。さらに必要なら `--blocks_to_swap` を 20→36 に増やす
- `gated` / 401 → HF トークンを確認（通常は不要。レート制限の可能性）
- 学習が途中で止まる → このセルを再実行（未完了ジョブだけ最初から。完了済みはスキップ）
- LoRA が効かない → `trigger.txt` のトリガーがキャプション先頭に入っているか、steps / lr を確認
- 途中チェックポイントが残らない → `--save_last_n_steps 2` により直近2個+最終のみ残る仕様です